# Imports

In [2]:
import torch
from torch import nn

# Parameters

# Data Collection

In [65]:
data = torch.randint(-10, 11, (1, 1000)).repeat(15, 1).float()
print(data)

tensor([[ 9., -6.,  2.,  ..., -8.,  3.,  8.],
        [ 9., -6.,  2.,  ..., -8.,  3.,  8.],
        [ 9., -6.,  2.,  ..., -8.,  3.,  8.],
        ...,
        [ 9., -6.,  2.,  ..., -8.,  3.,  8.],
        [ 9., -6.,  2.,  ..., -8.,  3.,  8.],
        [ 9., -6.,  2.,  ..., -8.,  3.,  8.]])


# Model Class Definition

In [9]:
class RNADE(nn.Module):
  def __init__(self, D, H, K=3):
    super().__init__()
    self.D = D
    self.H = H
    self.K = K
    self.W = nn.Parameter(torch.randn(H, D) * 0.01)
    self.c = nn.Parameter(torch.zeros(H))
    self.W_mu = nn.Parameter(torch.randn(D, K, H) * 0.01)
    self.b_mu = nn.Parameter(torch.zeros(D, K))
    self.W_si = nn.Parameter(torch.randn(D, K, H) * 0.01)
    self.b_si = nn.Parameter(torch.zeros(D, K))
    self.W_pi = nn.Parameter(torch.randn(D, K, H) * 0.01)
    self.b_pi = nn.Parameter(torch.zeros(D, K))
    self.min_si = 0.05

  def log_density(self, X):
    result = 0.0
    for i in range(self.D):
      h = torch.sigmoid(self.W[:, :i] @ X[:i, :] + self.c[:, None])
      mu = self.W_mu[i] @ h + self.b_mu[i, :, None]
      si = nn.functional.softplus(self.W_si[i] @ h + self.b_si[i, :, None]) + self.min_si
      log_pi = torch.log_softmax(self.W_pi[i] @ h + self.b_pi[i, :, None], dim=0)
      log_d = torch.distributions.Normal(mu, si).log_prob(X[i, :])
      result = result + torch.logsumexp(log_pi + log_d, dim=0)
    return result

  def forward(self, X):
    return self.log_density(X).exp()

  def fit(self, X, epochs=10000, lr=0.0003):
    self.train()
    optimizer = torch.optim.Adam(self.parameters(), lr=lr)
    for e in range(epochs):
      loss = -self.log_density(X).mean()
      optimizer.zero_grad()
      loss.backward()
      optimizer.step()
      if e % 1000 == 0:
        print(f"Epoch {e}: loss = {loss.item():.4f}")

  @torch.no_grad()
  def generate(self, n):
    X = torch.zeros(self.D, n, device=self.W.device, dtype=self.W.dtype)
    for i in range(self.D):
      h = torch.sigmoid(self.W[:, :i] @ X[:i, :] + self.c[:, None])
      mu = self.W_mu[i] @ h + self.b_mu[i, :, None]
      si = self.W_si[i] @ h + self.b_si[i, :, None]
      si = nn.functional.softplus(si) + self.min_si
      logits = self.W_pi[i] @ h + self.b_pi[i, :, None]
      k = torch.distributions.Categorical(logits=logits.T).sample()
      cols = torch.arange(n, device=self.W.device)
      X[i, :] = mu[k, cols] + si[k, cols] * torch.randn_like(X[i, :])
    return X

# Wrapper Class Definition

In [48]:
class AutoOptPricer():
  def __init__(self, model):
    self.model = model

  def generate_euro(self, current_price):
    paths = self.model.generate(500)
    movements = torch.prod((paths + 100) / 100, dim=0)
    prices = current_price * movements
    values = (prices - current_price).clamp(min=0)
    return values.mean(), paths

# Model Instantiation


In [ ]:
model = RNADE(D=15, H=32, K=5)
model.fit(data)

# Model Testing

In [ ]:
data = model.generate(n=8)

a = AutoOptPricer(model)
a.generate_euro(100)

In [ ]:
# ChatGPT Generated:
import torch
import matplotlib.pyplot as plt

a = AutoOptPricer(model)
p, movements = a.generate_euro(100)


movements = movements.float()

prices = 100 * torch.cumprod(1 + movements / 100, dim=0)
prices = torch.cat([
    prices.new_full((1, prices.shape[1]), 100),
    prices,
], dim=0)

plt.figure(figsize=(10, 6))
plt.plot(prices.detach().cpu().numpy(), alpha=0.6)
plt.xlabel("Movement step")
plt.ylabel("Stock price")
plt.title("Individual stock price paths")
plt.xticks(range(prices.shape[0]))
plt.grid(alpha=0.3)
plt.show()

print(p)